<a href="https://colab.research.google.com/github/srushtitelang18/Satellite-Land-Cover-Classification/blob/main/improved_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q osmnx geopandas pyproj rasterio matplotlib segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_PATH = "/content/drive/MyDrive/satellite_landcover"

In [4]:
import rasterio
import numpy as np
import osmnx as ox
from rasterio.features import rasterize
from pyproj import Transformer
import geopandas as gpd

IMG_PATH = "/content/drive/MyDrive/Sentinel_AI_Project.tif"

img = rasterio.open(IMG_PATH)
bounds = img.bounds
crs = img.crs
transform = img.transform
H, W = img.height, img.width

# Read bands
blue  = img.read(1).astype(np.float32)
green = img.read(2).astype(np.float32)
red   = img.read(3).astype(np.float32)
nir   = img.read(4).astype(np.float32)

# Indices
eps = 1e-6
ndvi = (nir - red) / (nir + red + eps)
ndwi = (green - nir) / (green + nir + eps)

veg = ndvi > 0.25
water = ndwi > 0.1

# OSM fetch
transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
west, south = transformer.transform(bounds.left, bounds.bottom)
east, north = transformer.transform(bounds.right, bounds.top)

cx, cy = (east+west)/2, (north+south)/2

buildings = ox.features_from_point((cy, cx), tags={'building': True}, dist=3000)
roads = ox.graph_to_gdfs(ox.graph_from_point((cy, cx), dist=3000), nodes=False)

# Reproject
buildings = buildings.to_crs(crs)
roads = roads.to_crs(crs)

# Rasterize
def rasterize_gdf(gdf):
    shapes = [(g, 1) for g in gdf.geometry if g is not None]
    return rasterize(shapes, out_shape=(H, W), transform=transform)

road_mask = rasterize_gdf(roads)
building_mask = rasterize_gdf(buildings)

# Final mask
mask = np.zeros((H, W), dtype=np.uint8)
mask[veg] = 1
mask[water] = 2
mask[building_mask == 1] = 3
mask[road_mask == 1] = 0

# Save
with rasterio.open("/content/drive/MyDrive/training_mask_osm.tif", "w",
    driver="GTiff",
    height=H,
    width=W,
    count=1,
    dtype=mask.dtype,
    crs=crs,
    transform=transform) as dst:

    dst.write(mask, 1)

print("✅ Mask created")

✅ Mask created


In [5]:
import rasterio
import numpy as np
import os

IMG_PATH = "/content/drive/MyDrive/Sentinel_AI_Project.tif"
MASK_PATH = "/content/drive/MyDrive/training_mask_osm.tif"

# Save tiles in Colab (temporary but fine for training)
os.makedirs("tiles/images", exist_ok=True)
os.makedirs("tiles/masks", exist_ok=True)

img = rasterio.open(IMG_PATH).read()
mask = rasterio.open(MASK_PATH).read(1)

tile_size = 128
count = 0

for y in range(0, mask.shape[0] - tile_size, tile_size):
    for x in range(0, mask.shape[1] - tile_size, tile_size):

        img_tile = img[:, y:y+tile_size, x:x+tile_size]
        mask_tile = mask[y:y+tile_size, x:x+tile_size]

        np.save(f"tiles/images/img_{count}.npy", img_tile)
        np.save(f"tiles/masks/mask_{count}.npy", mask_tile)

        count += 1

print("✅ Tiles created:", count)

✅ Tiles created: 81


In [6]:
!pip install -q segmentation-models-pytorch

In [7]:
import torch
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

# -----------------------------
# DEVICE (GPU)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# DATASET
# -----------------------------
class TileDataset(Dataset):
    def __init__(self):
        self.files = sorted(os.listdir("/content/tiles/images"))  # sorted for consistency

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = np.load(f"/content/tiles/images/{self.files[idx]}").astype(np.float32)
        mask = np.load(f"/content/tiles/masks/{self.files[idx].replace('img', 'mask')}")

        img = img / (img.max() + 1e-6)

        # ✅ FIX 1: Ensure mask is explicitly int64 BEFORE converting to tensor
        mask = mask.astype(np.int64)

        return torch.from_numpy(img), torch.from_numpy(mask)

# ✅ FIX 2: Actually instantiate dataset and loader (was missing!)
dataset = TileDataset()
loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)

# -----------------------------
# CHOOSE MODEL
# -----------------------------
MODEL_TYPE = "unet"   # change to "deeplab"

# ✅ FIX 3: dataset[0] now works because dataset is defined above
in_channels = dataset[0][0].shape[0]

if MODEL_TYPE == "unet":
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=in_channels,
        classes=4
    )
elif MODEL_TYPE == "deeplab":
    model = smp.DeepLabV3Plus(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=in_channels,
        classes=4
    )

model = model.to(device)

# -----------------------------
# LOSS + OPTIMIZER
# -----------------------------
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# -----------------------------
# TRAINING LOOP
# -----------------------------
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, masks in loader:
        imgs = imgs.to(device)
        masks = masks.to(device)  # will be torch.int64 ✅

        preds = model(imgs)       # shape: (B, 4, H, W)
        loss = loss_fn(preds, masks)  # masks must be (B, H, W) int64

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

# -----------------------------
# SAVE MODEL
# -----------------------------
torch.save(model.state_dict(), "/content/drive/MyDrive/model.pth")
print("✅ Model saved to Drive")

Using device: cuda
Epoch 1 Loss: 11.3257
Epoch 2 Loss: 8.1476
Epoch 3 Loss: 7.0502
Epoch 4 Loss: 6.9431
Epoch 5 Loss: 6.3662
✅ Model saved to Drive


In [8]:
# ─── Install deps ───────────────────────────────────────────────────────────
!pip install rasterio osmnx geopandas pyproj --quiet

import rasterio
import numpy as np
import osmnx as ox
from rasterio.features import rasterize
from pyproj import Transformer
import geopandas as gpd
from shapely.geometry import box

IMG_PATH = "/content/drive/MyDrive/Sentinel_AI_Project.tif"

# ─── Open image ─────────────────────────────────────────────────────────────
img = rasterio.open(IMG_PATH)
bounds = img.bounds
crs    = img.crs
transform = img.transform
H, W   = img.height, img.width

print(f"Image size: {H}x{W}, CRS: {crs}")

# ─── Read bands ─────────────────────────────────────────────────────────────
blue  = img.read(1).astype(np.float32)
green = img.read(2).astype(np.float32)
red   = img.read(3).astype(np.float32)
nir   = img.read(4).astype(np.float32)

eps  = 1e-6
ndvi = (nir - red)   / (nir + red   + eps)
ndwi = (green - nir) / (green + nir + eps)

veg   = ndvi > 0.25
water = ndwi > 0.1

print(f"Vegetation pixels: {veg.sum():,} | Water pixels: {water.sum():,}")

# ─── Convert bounds to WGS84 for OSM fetch ──────────────────────────────────
transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
west, south = transformer.transform(bounds.left,  bounds.bottom)
east, north = transformer.transform(bounds.right, bounds.top)
cx, cy = (east + west) / 2, (north + south) / 2

print(f"Fetching OSM data around ({cy:.4f}, {cx:.4f}) ...")

# ─── Fetch OSM features ─────────────────────────────────────────────────────
# FIX: use retain_all=True so disconnected roads are kept
buildings = ox.features_from_point((cy, cx), tags={'building': True}, dist=3000)
G         = ox.graph_from_point((cy, cx), dist=3000, retain_all=True)
roads     = ox.graph_to_gdfs(G, nodes=False)

# ─── Reproject to image CRS ─────────────────────────────────────────────────
buildings = buildings.to_crs(crs)
roads     = roads.to_crs(crs)

# ─── Clip to image extent (avoids artefacts outside the raster) ─────────────
img_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
buildings = buildings[buildings.geometry.intersects(img_bbox)]
roads     = roads[roads.geometry.intersects(img_bbox)]

print(f"Buildings: {len(buildings)} | Road segments: {len(roads)}")

# ─── Rasterize helper ───────────────────────────────────────────────────────
def rasterize_gdf(gdf):
    geoms = [g for g in gdf.geometry if g is not None and not g.is_empty]
    if not geoms:
        return np.zeros((H, W), dtype=np.uint8)
    shapes = [(g, 1) for g in geoms]
    return rasterize(shapes, out_shape=(H, W), transform=transform,
                     fill=0, dtype=np.uint8)

road_mask     = rasterize_gdf(roads)
building_mask = rasterize_gdf(buildings)

# ─── Build final mask ───────────────────────────────────────────────────────
# Class map:
#   0 = background / road (roads override everything)
#   1 = vegetation
#   2 = water
#   3 = building
mask = np.zeros((H, W), dtype=np.uint8)
mask[veg]              = 1
mask[water]            = 2
mask[building_mask==1] = 3
mask[road_mask==1]     = 0   # roads on top

unique, counts = np.unique(mask, return_counts=True)
for cls, cnt in zip(unique, counts):
    names = {0:"road/bg", 1:"vegetation", 2:"water", 3:"building"}
    print(f"  Class {cls} ({names[cls]}): {cnt:,} px  ({100*cnt/mask.size:.1f}%)")

# ─── Save mask ──────────────────────────────────────────────────────────────
MASK_OUT = "/content/drive/MyDrive/training_mask_osm.tif"
with rasterio.open(MASK_OUT, "w",
    driver="GTiff", height=H, width=W,
    count=1, dtype=mask.dtype, crs=crs, transform=transform) as dst:
    dst.write(mask, 1)

print(f"\n✅ Mask saved → {MASK_OUT}")

Image size: 1253x1278, CRS: EPSG:4326
Vegetation pixels: 500,066 | Water pixels: 5,971
Fetching OSM data around (12.9716, 77.5947) ...
Buildings: 44371 | Road segments: 47888
  Class 0 (road/bg): 1,011,489 px  (63.2%)
  Class 1 (vegetation): 450,911 px  (28.2%)
  Class 2 (water): 5,968 px  (0.4%)
  Class 3 (building): 132,966 px  (8.3%)

✅ Mask saved → /content/drive/MyDrive/training_mask_osm.tif


In [9]:
# ─── Install deps ───────────────────────────────────────────────────────────
!pip install rasterio --quiet

import rasterio
import numpy as np
import os

IMG_PATH  = "/content/drive/MyDrive/Sentinel_AI_Project.tif"
MASK_PATH = "/content/drive/MyDrive/training_mask_osm.tif"
TILE_SIZE = 128
MIN_VALID = 0.7   # discard tiles where >30% of pixels are nodata/black

os.makedirs("tiles/images", exist_ok=True)
os.makedirs("tiles/masks",  exist_ok=True)

img_src  = rasterio.open(IMG_PATH)
mask_src = rasterio.open(MASK_PATH)

img  = img_src.read()                    # (C, H, W)  float or uint16
mask = mask_src.read(1).astype(np.uint8) # (H, W)

print(f"Image shape: {img.shape}  |  Mask shape: {mask.shape}")
print(f"Image dtype: {img.dtype}  |  Mask classes: {np.unique(mask)}")

H, W  = mask.shape
count = 0
skipped = 0

for y in range(0, H - TILE_SIZE + 1, TILE_SIZE):
    for x in range(0, W - TILE_SIZE + 1, TILE_SIZE):

        img_tile  = img[:, y:y+TILE_SIZE, x:x+TILE_SIZE].astype(np.float32)
        mask_tile = mask[y:y+TILE_SIZE, x:x+TILE_SIZE]

        # FIX: skip near-empty tiles (all zeros = nodata)
        if img_tile.max() < 1e-6:
            skipped += 1
            continue

        # Normalize per-tile (safe with small epsilon)
        img_tile = img_tile / (img_tile.max() + 1e-6)

        np.save(f"tiles/images/img_{count}.npy",  img_tile)
        np.save(f"tiles/masks/mask_{count}.npy",  mask_tile)
        count += 1

print(f"\n✅ Tiles saved: {count}  |  Skipped (empty): {skipped}")
print(f"   Image channels: {img.shape[0]}  ← use this as in_channels in the model")

Image shape: (8, 1253, 1278)  |  Mask shape: (1253, 1278)
Image dtype: float32  |  Mask classes: [0 1 2 3]

✅ Tiles saved: 81  |  Skipped (empty): 0
   Image channels: 8  ← use this as in_channels in the model


In [ ]:
# ─── Install deps ────────────────────────────────────────────────────────────
!pip install segmentation-models-pytorch --quiet

import torch
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp

# ─── Device ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ─── Dataset ─────────────────────────────────────────────────────────────────
class TileDataset(Dataset):
    def __init__(self, img_dir="tiles/images", mask_dir="tiles/masks"):
        self.img_dir  = img_dir
        self.mask_dir = mask_dir
        self.files    = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        img  = np.load(f"{self.img_dir}/{name}").astype(np.float32)
        mask = np.load(f"{self.mask_dir}/{name.replace('img', 'mask')}")

        img  = img / (img.max() + 1e-6)
        mask = mask.astype(np.int64)   # required by CrossEntropyLoss

        return torch.from_numpy(img), torch.from_numpy(mask)

# ─── Instantiate dataset ─────────────────────────────────────────────────────
dataset = TileDataset()
print(f"Total tiles: {len(dataset)}")

sample_img, sample_mask = dataset[0]
IN_CHANNELS = sample_img.shape[0]

# ✅ FIX: auto-detect number of classes from actual mask values
all_classes = set()
for i in range(len(dataset)):
    _, m = dataset[i]
    all_classes.update(m.numpy().flatten().tolist())
NUM_CLASSES = len(all_classes)

print(f"Input channels : {IN_CHANNELS}")
print(f"Mask classes   : {sorted(all_classes)}  →  NUM_CLASSES = {NUM_CLASSES}")

# ─── Train / val split ───────────────────────────────────────────────────────
val_size   = max(1, int(0.1 * len(dataset)))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=8,  shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8,  shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {train_size} tiles  |  Val: {val_size} tiles")

# ─── Model ───────────────────────────────────────────────────────────────────
MODEL_TYPE = "unet"   # or "deeplab"

if MODEL_TYPE == "unet":
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
    )
elif MODEL_TYPE == "deeplab":
    model = smp.DeepLabV3Plus(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
    )

model = model.to(device)
print(f"Model : {MODEL_TYPE}  |  Params: {sum(p.numel() for p in model.parameters()):,}")

# ─── Class weights (handles imbalanced land-cover) ───────────────────────────
# Count pixels per class across all masks
print("Computing class weights ...")
class_counts = np.zeros(NUM_CLASSES, dtype=np.float64)
for i in range(len(dataset)):
    _, m = dataset[i]
    for c in range(NUM_CLASSES):
        class_counts[c] += (m.numpy() == c).sum()

# Inverse-frequency weighting
total = class_counts.sum()
class_weights = total / (NUM_CLASSES * class_counts + 1e-6)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights: {class_weights.cpu().numpy().round(3)}")

# ─── Loss + optimiser ────────────────────────────────────────────────────────
loss_fn   = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ✅ FIX: removed verbose=True (dropped in PyTorch 2.4+)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# ─── Metric helper (mean IoU) ─────────────────────────────────────────────────
def mean_iou(preds, masks, num_classes):
    """preds: (B,H,W) int  |  masks: (B,H,W) int"""
    ious = []
    for c in range(num_classes):
        tp = ((preds == c) & (masks == c)).sum().item()
        fp = ((preds == c) & (masks != c)).sum().item()
        fn = ((preds != c) & (masks == c)).sum().item()
        denom = tp + fp + fn
        ious.append(tp / denom if denom > 0 else float('nan'))
    valid = [v for v in ious if not np.isnan(v)]
    return np.mean(valid) if valid else 0.0

# ─── Training loop ───────────────────────────────────────────────────────────
EPOCHS        = 20
best_val_loss = float("inf")
best_epoch    = 0
history       = {"train_loss": [], "val_loss": [], "val_iou": []}

print("\n── Training ─────────────────────────────────────────────────────────")

for epoch in range(EPOCHS):

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for imgs, masks in train_loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)          # int64, shape (B, H, W)

        preds = model(imgs)               # (B, NUM_CLASSES, H, W)
        loss  = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    val_iou  = 0.0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)

            preds     = model(imgs)
            val_loss += loss_fn(preds, masks).item()

            pred_cls  = preds.argmax(dim=1)   # (B, H, W)
            val_iou  += mean_iou(pred_cls, masks, NUM_CLASSES)

    val_loss /= len(val_loader)
    val_iou  /= len(val_loader)

    # ── LR scheduler step (replaces verbose=True) ────────────────────────────
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    lr_tag = f"  ↓ LR {old_lr:.6f}→{new_lr:.6f}" if new_lr < old_lr else ""

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train loss: {train_loss:.4f} | "
          f"Val loss: {val_loss:.4f} | "
          f"Val mIoU: {val_iou:.4f} | "
          f"LR: {new_lr:.6f}{lr_tag}")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)

    # ── Save best model ───────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch + 1
        torch.save({
            "epoch"       : epoch + 1,
            "model_state" : model.state_dict(),
            "optimizer"   : optimizer.state_dict(),
            "val_loss"    : val_loss,
            "val_iou"     : val_iou,
            "num_classes" : NUM_CLASSES,
            "in_channels" : IN_CHANNELS,
            "model_type"  : MODEL_TYPE,
        }, "/content/drive/MyDrive/model_best.pth")
        print(f"  ✅ Best model saved  (epoch {best_epoch}, val loss {best_val_loss:.4f})")

# ─── Save final model ─────────────────────────────────────────────────────────
torch.save({
    "epoch"       : EPOCHS,
    "model_state" : model.state_dict(),
    "num_classes" : NUM_CLASSES,
    "in_channels" : IN_CHANNELS,
    "model_type"  : MODEL_TYPE,
}, "/content/drive/MyDrive/model_final.pth")
print(f"\n✅ Final model saved")
print(f"   Best epoch: {best_epoch}  |  Best val loss: {best_val_loss:.4f}")

# ─── Plot training curves ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history["train_loss"], label="Train loss")
ax1.plot(history["val_loss"],   label="Val loss")
ax1.axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f"Best (ep {best_epoch})")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss curves"); ax1.legend()

ax2.plot(history["val_iou"], color='green', label="Val mIoU")
ax2.axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("mIoU")
ax2.set_title("Validation mIoU"); ax2.legend()

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/training_curves.png", dpi=120)
plt.show()
print("✅ Training curves saved")

In [ ]:
# ─── Install deps ────────────────────────────────────────────────────────────
!pip install segmentation-models-pytorch rasterio matplotlib --quiet

import torch
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import segmentation_models_pytorch as smp
import os
import warnings
warnings.filterwarnings("ignore")

# ─── Config ──────────────────────────────────────────────────────────────────
MODEL_PATH = "/content/drive/MyDrive/model_best.pth"
IMG_PATH   = "/content/drive/MyDrive/Sentinel_AI_Project.tif"
OUT_DIR    = "/content/drive/MyDrive/predictions"
TILE_SIZE  = 128
ALPHA      = 0.45

os.makedirs(OUT_DIR, exist_ok=True)

# ─── Device ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ─── Load image info ─────────────────────────────────────────────────────────
with rasterio.open(IMG_PATH) as src:
    IN_CHANNELS = src.count
    H_full, W_full = src.height, src.width
    print(f"Image: {IN_CHANNELS} bands, {H_full}×{W_full} px")

# ─── Load checkpoint ─────────────────────────────────────────────────────────
# ✅ FIX 1: weights_only=False  (PyTorch 2.6 changed default to True)
checkpoint  = torch.load(MODEL_PATH, map_location=device, weights_only=False)

# ✅ FIX 2: read config saved during training
IN_CHANNELS = checkpoint["in_channels"]   # 8
NUM_CLASSES = checkpoint["num_classes"]   # 3
MODEL_TYPE  = checkpoint["model_type"]    # "unet"

print(f"Checkpoint → in_channels={IN_CHANNELS}, num_classes={NUM_CLASSES}, model={MODEL_TYPE}")

# ─── Build model ─────────────────────────────────────────────────────────────
if MODEL_TYPE == "unet":
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
    )
elif MODEL_TYPE == "deeplab":
    model = smp.DeepLabV3Plus(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
    )

# ✅ FIX 3: load only the weights dict, not the whole checkpoint
model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
model.eval()
print("✅ Model loaded")

# ─── Class colors + names (3 classes) ────────────────────────────────────────
# ─── Class colors + names (4 classes — matches your checkpoint) ──────────────
CLASS_COLORS = np.array([
    [80,  80,  80,  255],   # 0 road/background — gray
    [56, 168,  0,   255],   # 1 vegetation      — green
    [0,  112, 255,  255],   # 2 water            — blue
    [215, 75,  0,   255],   # 3 building         — orange-red
], dtype=np.uint8)

CLASS_NAMES = ["Road / background", "Vegetation", "Water", "Building"]

# ─── Load full image ─────────────────────────────────────────────────────────
with rasterio.open(IMG_PATH) as src:
    img_full  = src.read().astype(np.float32)   # (C, H, W)
    profile   = src.profile
    transform = src.transform
    crs       = src.crs

# ─── Pad image so it divides evenly into tiles ───────────────────────────────
C, H, W = img_full.shape
pad_h = (TILE_SIZE - H % TILE_SIZE) % TILE_SIZE
pad_w = (TILE_SIZE - W % TILE_SIZE) % TILE_SIZE

img_padded = np.pad(img_full, ((0,0),(0,pad_h),(0,pad_w)), mode='reflect')
pred_full  = np.zeros((img_padded.shape[1], img_padded.shape[2]), dtype=np.uint8)

Hp, Wp = img_padded.shape[1], img_padded.shape[2]
total_tiles = (Hp // TILE_SIZE) * (Wp // TILE_SIZE)
done = 0

print(f"Running inference on {total_tiles} tiles ...")

# ─── Tile-based inference ─────────────────────────────────────────────────────
with torch.no_grad():
    for y in range(0, Hp, TILE_SIZE):
        for x in range(0, Wp, TILE_SIZE):
            tile = img_padded[:, y:y+TILE_SIZE, x:x+TILE_SIZE]
            tile = tile / (tile.max() + 1e-6)

            tile_t = torch.from_numpy(tile).unsqueeze(0).to(device)  # (1,C,128,128)
            logits = model(tile_t)                                    # (1,3,128,128)
            pred   = logits.argmax(dim=1).squeeze(0)                  # (128,128)
            pred_full[y:y+TILE_SIZE, x:x+TILE_SIZE] = pred.cpu().numpy()

            done += 1
            if done % 50 == 0 or done == total_tiles:
                print(f"  {done}/{total_tiles} tiles done", end="\r")

# Crop back to original size
pred_full = pred_full[:H, :W]
print(f"\n✅ Inference complete. Shape: {pred_full.shape}")

# ─── Class distribution ───────────────────────────────────────────────────────
print("\nPredicted class distribution:")
unique, counts = np.unique(pred_full, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls} ({CLASS_NAMES[cls]}): {cnt:,} px  ({100*cnt/pred_full.size:.1f}%)")

# ─── Helper: mask → RGBA ──────────────────────────────────────────────────────
def mask_to_rgba(mask, colors):
    rgba = np.zeros((*mask.shape, 4), dtype=np.uint8)
    for cls_id, color in enumerate(colors):
        rgba[mask == cls_id] = color
    return rgba

pred_rgba = mask_to_rgba(pred_full, CLASS_COLORS)

# ─── Helper: Sentinel bands → RGB preview ────────────────────────────────────
def sentinel_to_rgb(img, r=2, g=1, b=0):
    rgb = img[[r, g, b]].transpose(1, 2, 0)
    p2, p98 = np.percentile(rgb, (2, 98))
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
    return (rgb * 255).astype(np.uint8)

rgb_preview = sentinel_to_rgb(img_full)

# ─── Legend patches ───────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=CLASS_COLORS[i][:3] / 255, label=CLASS_NAMES[i])
    for i in range(NUM_CLASSES)
]

# ─── Output 1: Pure prediction map ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(pred_rgba)
ax.set_title("Segmentation prediction", fontsize=14, fontweight='bold')
ax.axis("off")
ax.legend(handles=legend_patches, loc="lower right",
          fontsize=10, framealpha=0.85, edgecolor='gray')
out1 = f"{OUT_DIR}/prediction_map.png"
fig.savefig(out1, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f"✅ Saved: {out1}")

# ─── Output 2: Overlay ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(rgb_preview)
ax.imshow(pred_rgba, alpha=ALPHA)
ax.set_title("Overlay: RGB + prediction", fontsize=14, fontweight='bold')
ax.axis("off")
ax.legend(handles=legend_patches, loc="lower right",
          fontsize=10, framealpha=0.85, edgecolor='gray')
out2 = f"{OUT_DIR}/overlay.png"
fig.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f"✅ Saved: {out2}")

# ─── Output 3: Side-by-side comparison ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 10))
axes[0].imshow(rgb_preview)
axes[0].set_title("Original (RGB)", fontsize=14, fontweight='bold')
axes[0].axis("off")
axes[1].imshow(pred_rgba)
axes[1].set_title("Model prediction", fontsize=14, fontweight='bold')
axes[1].axis("off")
axes[1].legend(handles=legend_patches, loc="lower right",
               fontsize=10, framealpha=0.85, edgecolor='gray')
plt.tight_layout(pad=1.5)
out3 = f"{OUT_DIR}/comparison.png"
fig.savefig(out3, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f"✅ Saved: {out3}")

# ─── Output 4: Per-class confidence maps ─────────────────────────────────────
print("\nGenerating confidence maps ...")
prob_full = np.zeros((NUM_CLASSES, H, W), dtype=np.float32)

with torch.no_grad():
    for y in range(0, Hp, TILE_SIZE):
        for x in range(0, Wp, TILE_SIZE):
            tile   = img_padded[:, y:y+TILE_SIZE, x:x+TILE_SIZE]
            tile   = tile / (tile.max() + 1e-6)
            tile_t = torch.from_numpy(tile).unsqueeze(0).to(device)
            logits = model(tile_t)
            probs  = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

            y_end = min(y + TILE_SIZE, H)
            x_end = min(x + TILE_SIZE, W)
            prob_full[:, y:y_end, x:x_end] = probs[:, :y_end-y, :x_end-x]

cmaps = ['Greys_r', 'Greens', 'Blues', 'Oranges']
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(18, 6))
for i in range(NUM_CLASSES):
    im = axes[i].imshow(prob_full[i], cmap=cmaps[i], vmin=0, vmax=1)
    axes[i].set_title(f"P({CLASS_NAMES[i]})", fontsize=11, fontweight='bold')
    axes[i].axis("off")
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

plt.suptitle("Per-class prediction confidence", fontsize=13, fontweight='bold')
plt.tight_layout()
out4 = f"{OUT_DIR}/confidence_maps.png"
fig.savefig(out4, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f"✅ Saved: {out4}")

# ─── Output 5: GeoTIFF (opens in QGIS / ArcGIS) ──────────────────────────────
profile.update(count=1, dtype=rasterio.uint8, compress='lzw')
out5 = f"{OUT_DIR}/prediction.tif"
with rasterio.open(out5, "w", **profile) as dst:
    dst.write(pred_full.astype(np.uint8), 1)
print(f"✅ Saved GeoTIFF: {out5}")

# ─── Summary ─────────────────────────────────────────────────────────────────
print("\n── All outputs saved to Drive ───────────────────────────────────────")
for path, label in [
    (out1, "Prediction map  "),
    (out2, "Overlay         "),
    (out3, "Comparison      "),
    (out4, "Confidence maps "),
    (out5, "GeoTIFF         "),
]:
    print(f"  {label} → {path}")